## 3. XGBoost Baseline

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

IN_COLAB=False  REPO_ROOT=/Users/ucabtt4/Projects/Dissertation/dissertation


In [11]:
import json

import numpy as np

from src.dataloader import load_cgm_stats, load_arrays, load_participant_ids
from src import metrics
from src.models.classical_baseline import XGBoostPredictor, HORIZONS

MODEL_NAME = 'XGBoost'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'xgboost'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'xgboost_stability_summary.json'

In [4]:
stats = load_cgm_stats()
X_train, y_train = load_arrays('train')
X_val, y_val = load_arrays('val')
X_test, y_test = load_arrays('test')

test_participant_ids = load_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print('train set shape: ', X_train.shape, '\nvaluation set shape: ', X_val.shape, '\ntest set shape: ', X_test.shape)
print('\nnumber of test participants: ', len(PARTICIPANT_IDS))
print('Corresponding participants ID: \n', PARTICIPANT_IDS)

train set shape:  (83532, 24) 
valuation set shape:  (16739, 24) 
test set shape:  (16842, 24)

number of test participants:  6
Corresponding participants ID: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]


In [7]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for seed in SEEDS:
    print(f"\n{'-' * 32}seed={seed}{'-' * 32}")

    model = XGBoostPredictor(random_state=seed).fit(X_train, y_train)

    val_pred, test_pred = model.predict(X_val), model.predict(X_test)
    val_results = metrics.evaluate(val_pred, y_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(test_pred, y_test, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        test_pred, y_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed,
        'val_results': val_results, 'test_results': test_results,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'xgboost_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    test_rmse = [round(test_results[h]['rmse'], 2) for h in HORIZONS]
    print(f"seed={seed:>3}  test RMSE @ {HORIZONS} min = {test_rmse}")


----------------seed=7----------------
seed=  7  test RMSE @ [15, 30, 60, 90, 120] min = [11.83, 19.12, 27.6, 32.63, 35.51]

----------------seed=14----------------
seed= 14  test RMSE @ [15, 30, 60, 90, 120] min = [11.87, 19.11, 27.61, 32.67, 35.47]

----------------seed=21----------------
seed= 21  test RMSE @ [15, 30, 60, 90, 120] min = [11.89, 19.14, 27.58, 32.66, 35.51]

----------------seed=28----------------
seed= 28  test RMSE @ [15, 30, 60, 90, 120] min = [11.82, 19.11, 27.58, 32.61, 35.48]

----------------seed=35----------------
seed= 35  test RMSE @ [15, 30, 60, 90, 120] min = [11.82, 19.12, 27.6, 32.68, 35.49]

----------------seed=42----------------
seed= 42  test RMSE @ [15, 30, 60, 90, 120] min = [11.86, 19.17, 27.6, 32.65, 35.5]

----------------seed=49----------------
seed= 49  test RMSE @ [15, 30, 60, 90, 120] min = [11.82, 19.1, 27.61, 32.6, 35.47]

----------------seed=56----------------
seed= 56  test RMSE @ [15, 30, 60, 90, 120] min = [11.86, 19.14, 27.63, 32.68

In [12]:
# Test Results: overall mean +/- SD_seed (SD_participant)
def _extract(participant_result, metric):

    """Extract a metric from one participant's result."""
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']
summary = {}

for h in HORIZONS:

    summary[h] = {}

    for m in METRICS:
        if m == 'zone_a':
            seed_values = [
                r['test_results'][h]['ceg']['A']
            for r in all_results
            ]
        elif m == 'zone_ab':
            seed_values = [
                r['test_results'][h]['ceg']['A']
                + r['test_results'][h]['ceg']['B']
                for r in all_results
            ]
        else:
            seed_values = [
                r['test_results'][h][m]
                for r in all_results
            ]

        mean = float(np.mean(seed_values))
        sd1 = float(np.std(seed_values, ddof=1))

        participant_means = []

        for pid in PARTICIPANT_IDS:
            values_across_seeds = [
                _extract(r['test_results_per_participant'][pid][h], m)
                for r in all_results
            ]
            participant_means.append(np.mean(values_across_seeds))
        sd2 = float(np.std(participant_means, ddof=1))
        summary[h][m] = {
            'mean': mean,
            'sd1_across_seeds': sd1,
            'sd2_across_participants': sd2,
        }

# ---------------------------------------------------------

# Print: mean +/- sd1 (sd2)

# ---------------------------------------------------------

print(
    f'=== {MODEL_NAME} test set: '
    f'overall mean +/- SD_seed (SD_participant) ==='
)

for h in HORIZONS:
    s = summary[h]
    print(
        f"{h:>3}-min:  "
        f"RMSE = {s['rmse']['mean']:.2f} +/- "
        f"{s['rmse']['sd1_across_seeds']:.2f} "
        f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
        f"MAE = {s['mae']['mean']:.2f} +/- "
        f"{s['mae']['sd1_across_seeds']:.2f} "
        f"({s['mae']['sd2_across_participants']:.2f})   "
        f"gRMSE = {s['grmse']['mean']:.2f} +/- "
        f"{s['grmse']['sd1_across_seeds']:.2f} "
        f"({s['grmse']['sd2_across_participants']:.2f})   "
        f"Zone A = {s['zone_a']['mean']:.2f} +/- "
        f"{s['zone_a']['sd1_across_seeds']:.2f} "
        f"({s['zone_a']['sd2_across_participants']:.2f})%   "
        f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- "
        f"{s['zone_ab']['sd1_across_seeds']:.2f} "
        f"({s['zone_ab']['sd2_across_participants']:.2f})%"
    )
# ---------------------------------------------------------

# Save one final summary JSON

# ---------------------------------------------------------

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': summary,
    }, f, indent=2)
print(f'\nsaved aggregate summary to {STABILITY_SUMMARY_PATH}')

=== XGBoost test set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 11.85 +/- 0.02 (2.48) mg/dL   MAE = 7.79 +/- 0.01 (1.74)   gRMSE = 13.39 +/- 0.03 (4.04)   Zone A = 96.90 +/- 0.04 (1.75)%   Zone A+B = 99.84 +/- 0.01 (0.22)%
 30-min:  RMSE = 19.12 +/- 0.02 (4.52) mg/dL   MAE = 12.47 +/- 0.01 (2.95)   gRMSE = 22.43 +/- 0.03 (7.45)   Zone A = 90.19 +/- 0.04 (3.32)%   Zone A+B = 99.52 +/- 0.01 (0.59)%
 60-min:  RMSE = 27.60 +/- 0.01 (9.42) mg/dL   MAE = 18.05 +/- 0.01 (5.52)   gRMSE = 33.90 +/- 0.02 (14.84)   Zone A = 81.66 +/- 0.05 (5.08)%   Zone A+B = 99.00 +/- 0.01 (0.94)%
 90-min:  RMSE = 32.64 +/- 0.03 (12.94) mg/dL   MAE = 21.66 +/- 0.02 (7.65)   gRMSE = 40.83 +/- 0.04 (20.20)   Zone A = 75.74 +/- 0.08 (6.76)%   Zone A+B = 98.44 +/- 0.03 (1.56)%
120-min:  RMSE = 35.49 +/- 0.01 (15.13) mg/dL   MAE = 23.78 +/- 0.02 (9.14)   gRMSE = 44.70 +/- 0.02 (23.49)   Zone A = 71.81 +/- 0.07 (7.21)%   Zone A+B = 98.01 +/- 0.02 (2.37)%

saved aggregate summary to /Users/ucabtt4/